# 04 — Log sequence creation (LSTM-ready)

Read `*_processed.csv` files, slide a fixed-length window over **time-ordered** events, and save:

- Per building: `{stem}_X.npy`, `{stem}_y.npy`, `{stem}_meta.csv` (labels from `anomaly_link`)
- `sequence_manifest.json` (feature list, shapes, paths to each building’s arrays)

**By default** only `src.config.BUILDINGS` are included (sensor study cohort). Set `process_all_buildings=True` when scaling.

Tensor shape: **`(num_sequences, sequence_length, num_features)`** — ready for Keras/TensorFlow LSTM input.

In [9]:
import os
import sys

import numpy as np

SCRIPTS = os.path.abspath(os.path.join(os.getcwd(), "..", "scripts"))
if SCRIPTS not in sys.path:
    sys.path.insert(0, SCRIPTS)

from log_pipeline.preprocessing import LSTM_FEATURE_COLUMNS
from log_pipeline.sequences import run_sequence_pipeline

LOGS_PROCESSED = os.path.abspath(os.path.join(os.getcwd(), "..", "data", "processed", "logs_processed"))
SEQ_OUT = os.path.abspath(os.path.join(os.getcwd(), "..", "data", "processed", "log_sequences"))

os.makedirs(SEQ_OUT, exist_ok=True)

SEQUENCE_LENGTH = 10
STRIDE = 1
LABEL_MODE = "any"  # 'any' | 'last' | 'none'

# Must match numeric columns present in processed CSV (default from preprocessing).
feature_cols = list(LSTM_FEATURE_COLUMNS)
# If you ran 03 with encode_message=True, append "message_id" here.

print("Processed logs:", LOGS_PROCESSED)
print("Sequence output:", SEQ_OUT)
print("feature_cols:", feature_cols)

Processed logs: D:\Uni\Sem 4\PBL 1\smart-building-anomaly-detection\data\processed\logs_processed
Sequence output: D:\Uni\Sem 4\PBL 1\smart-building-anomaly-detection\data\processed\log_sequences
feature_cols: ['event_type', 'event_code', 'severity', 'device_id', 'subsystem', 'building_id', 'value']


In [11]:
manifest = run_sequence_pipeline(
    LOGS_PROCESSED,
    SEQ_OUT,
    feature_cols=feature_cols,
    sequence_length=SEQUENCE_LENGTH,
    stride=STRIDE,
    label_mode=LABEL_MODE,
    process_all_buildings=False,
)

print("manifest_path:", manifest.get("manifest_path"))
per_file = manifest.get("per_file") or []
print("buildings in manifest:", len(per_file))
if per_file:
    X0 = np.load(per_file[0]["X_path"])
    y0 = np.load(per_file[0]["y_path"])
    print("example:", per_file[0].get("building"), "X", X0.shape, "y", y0.shape)

manifest_path: D:\Uni\Sem 4\PBL 1\smart-building-anomaly-detection\data\processed\log_sequences\sequence_manifest.json
combined_X: [37607, 10, 7]
combined_y: [37607]


In [13]:
import json

mf = os.path.join(SEQ_OUT, "sequence_manifest.json")
if os.path.isfile(mf):
    with open(mf, encoding="utf-8") as f:
        man = json.load(f)
    items = man.get("per_file") or []
    if items:
        xp = items[0]["X_path"]
        yp = items[0]["y_path"]
        X = np.load(xp)
        y = np.load(yp)
        print("Loaded from manifest:", items[0].get("building"))
        print("X dtype:", X.dtype, "shape:", X.shape, "-> (N, timesteps, features)")
        print("y shape:", y.shape, "positive rate:", float(y.mean()))
    else:
        print("Manifest has no per_file entries.")
else:
    xp = os.path.join(SEQ_OUT, "log_sequences_X.npy")
    if os.path.isfile(xp):
        X = np.load(xp)
        print("X dtype:", X.dtype, "shape:", X.shape, "-> (N, timesteps, features)")
        yp = os.path.join(SEQ_OUT, "log_sequences_y.npy")
        if os.path.isfile(yp):
            y = np.load(yp)
            print("y shape:", y.shape, "positive rate:", float(y.mean()))
    else:
        print("No sequence_manifest.json — run the cell above first.")

X dtype: float32 shape: (37607, 10, 7) -> (N, timesteps, features)
y shape: (37607,) positive rate: 0.7928577126598771
